In [2]:
import torch
from d2l import torch as d2l
from torch import nn

In [ ]:

def corr2d_mulit_in(X,K):
    return sum(d2l.corr2d(x,k) for x,k in zip(X,K)) 
X = torch.tensor([[[0.0,1.0,2.0],[3.0,4.0,5.0],[6.0,7.0,8.0]],
                  [[1.0,2.0,3.0],[4.0,5.0,6.0],[7.0,8.0,9.0]]])
print(f'X:\n{X.shape} \n {X}')

K = torch.tensor([[[0.0,1.0],[2.0,3.0]],[[1.0,2.0],[3.0,4.0]]])
print(f'K:\n {K.shape} \n {K}')

print(corr2d_mulit_in(X,K))


# 多输出通道运算
def corr2d_multi_in_out(X,K):  # X为3通道矩阵，K为4通道矩阵，最外面维为输出通道      
    return torch.stack([corr2d_multi_in(X,k) for k in K],0) # 大k中每个小k是一个3D的Tensor。0表示stack堆叠函数里面在0这个维度堆叠。           

print(K.shape)
print((K+1).shape)
print((K+2).shape)
print(K)
print(K+1)
K = torch.stack((K, K+1, K+2),0) # K与K+1之间的区别为K的每个元素加1
print(K.shape)
print(corr2d_multi_in_out(X,K))

X:
torch.Size([2, 3, 3]) 
 tensor([[[0., 1., 2.],
         [3., 4., 5.],
         [6., 7., 8.]],

        [[1., 2., 3.],
         [4., 5., 6.],
         [7., 8., 9.]]])
K:
 torch.Size([2, 2, 2]) 
 tensor([[[0., 1.],
         [2., 3.]],

        [[1., 2.],
         [3., 4.]]])
tensor([[ 56.,  72.],
        [104., 120.]])


In [4]:
help(d2l.corr2d)

Help on function corr2d in module d2l.torch:

corr2d(X, K)
    Compute 2D cross-correlation.
    
    Defined in :numref:`sec_conv_layer`



In [4]:
# 1*1卷积
X = torch.normal(0,1,(3,3,3))   # norm函数生成0到1之间的(3,3,3)矩阵 
K = torch.normal(0,1,(2,3,1,1)) # 输出通道是2，输入通道是3，核是1X1
print(f'X \n {X.shape}{X}')
# 1×1卷积的多输入、多输出通道运算
def corr2d_multi_in_out_1x1(X,K):
    c_i, h, w = X.shape # 输入的通道数、宽、高
    c_o = K.shape[0]    # 输出的通道数
    X = X.reshape((c_i, h * w)) # 拉平操作，每一行表示一个通道的特征
    K = K.reshape((c_o,c_i)) 
    Y = torch.matmul(K,X) 
    return Y.reshape((c_o, h, w))

Y1 = corr2d_multi_in_out_1x1(X,K)
Y2 = corr2d_multi_in_out(X,K)
assert float(torch.abs(Y1-Y2).sum()) < 1e-6
print(float(torch.abs(Y1-Y2).sum()))

X 
 torch.Size([3, 3, 3])tensor([[[-2.1405,  2.3667, -1.1257],
         [-0.3075,  0.7065,  1.3460],
         [ 0.1035,  1.1677, -0.8387]],

        [[ 1.6161,  2.4033, -0.8372],
         [ 0.4252, -0.7042, -0.3482],
         [-0.8123,  1.1567,  0.8334]],

        [[-0.8510,  0.2835,  0.3401],
         [ 1.3481,  0.8441,  0.1064],
         [-0.1994,  0.0385,  1.1021]]])


NameError: name 'corr2d_multi_in_out' is not defined

In [5]:
def comp_conv2d(conv2d, X): # conv2d 作为传参传进去，在内部使用
    X = X.reshape((1,1)+X.shape) # 在维度前面加入一个通道数和批量大小数
    Y = conv2d(X)  # 卷积处理是一个四维的矩阵
    return Y.reshape(Y.shape[2:]) # 将前面两个维度拿掉

X = torch.rand(size=(8,8))
conv2d = nn.Conv2d(1,1,kernel_size=3,padding=1,stride=2) # Pytorch里面卷积函数的第一个参数为输出通道，第二个参数为输入通道   
print(comp_conv2d(conv2d,X).shape) 

conv2d = nn.Conv2d(1,1,kernel_size=(3,5),padding=(0,1),stride=(3,4)) # 一个稍微复杂的例子
print(comp_conv2d(conv2d,X).shape)

torch.Size([4, 4])
torch.Size([2, 2])
